<a href="https://colab.research.google.com/github/NandiniN2215/BIS_LAB/blob/main/Cuckoo_search_Network_intrusion_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import time
import math

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

np.random.seed(42)

# 1. Generate Synthetic Network Dataset

n_samples = 3000
n_features = 15

X = np.random.rand(n_samples, n_features)

score = X[:, 0] + X[:, 1] + X[:, 2] + X[:, 3] + X[:, 4]
y = (score > 2.5).astype(int)

print("Samples:", n_samples)
print("Features:", n_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25,
    random_state=42, stratify=y
)

# 2. Cuckoo Search Parameters

n_nests = 10
iterations = 15
alpha = 0.01
pa = 0.25
beta = 1.5

start = time.time()

# Initialize nests
nests = np.random.rand(n_nests, n_features)

# Sigmoid transfer function
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -50, 50)))

# Convert continuous position to binary mask
def binary_solution(position):
    return (np.random.rand(n_features) < sigmoid(position)).astype(int)

# Levy flight calculation
def levy_flight(size):
    sigma = (
        math.gamma(1 + beta) *
        math.sin(math.pi * beta / 2) /
        (
            math.gamma((1 + beta) / 2) *
            beta * 2 ** ((beta - 1) / 2)
        )
    ) ** (1 / beta)

    u = np.random.normal(0, sigma, size)
    v = np.random.normal(0, 1, size)

    return u / (np.abs(v) ** (1 / beta) + 1e-10)

# 3. Fitness Function

def fitness(mask):

    selected = np.where(mask == 1)[0]

    if len(selected) == 0:
        return 1.0

    model = RandomForestClassifier(
        n_estimators=50,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train[:, selected], y_train)
    pred = model.predict(X_test[:, selected])

    f1 = f1_score(y_test, pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(
        y_test, pred, labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    feature_ratio = len(selected) / n_features

    return (
        0.65 * (1 - f1)
        + 0.25 * fpr
        + 0.10 * feature_ratio
    )

# Initialize binary nests and fitness
binary_nests = np.array([
    binary_solution(n) for n in nests
])

scores = np.array([
    fitness(n) for n in binary_nests
])

best_index = np.argmin(scores)
best_position = nests[best_index].copy()
best_binary = binary_nests[best_index].copy()
best_score = scores[best_index]

# 4. Cuckoo Search Optimization

print("\nStarting Binary Cuckoo Search...\n")

for t in range(iterations):

    for i in range(n_nests):

        # Lévy flight position update:
        # x_i(t+1) = x_i(t) + alpha * Levy(lambda)

        step = levy_flight(n_features)

        new_position = nests[i] + alpha * step

        # Sigmoid transformation and binary conversion
        new_binary = binary_solution(new_position)

        new_score = fitness(new_binary)

        # Replace a randomly selected nest if better
        j = np.random.randint(n_nests)

        if new_score < scores[j]:
            nests[j] = new_position
            binary_nests[j] = new_binary
            scores[j] = new_score

        # Update global best
        if new_score < best_score:
            best_score = new_score
            best_position = new_position.copy()
            best_binary = new_binary.copy()

    # Abandon fraction Pa of worst nests
    abandon_count = max(1, int(pa * n_nests))
    worst_nests = np.argsort(scores)[-abandon_count:]

    for j in worst_nests:

        nests[j] = np.random.rand(n_features)
        binary_nests[j] = binary_solution(nests[j])
        scores[j] = fitness(binary_nests[j])

        if scores[j] < best_score:
            best_score = scores[j]
            best_position = nests[j].copy()
            best_binary = binary_nests[j].copy()

    print(
        f"Iteration {t+1}/{iterations} | "
        f"Best Fitness: {best_score:.5f} | "
        f"Selected Features: {np.sum(best_binary)}"
    )

# 5. Optimized Feature Selection

selected = np.where(best_binary == 1)[0]

if len(selected) == 0:
    selected = np.array([np.argmax(best_position)])

print("\nSelected Feature Indices:", selected + 1)

print("Original Features:", n_features)
print("Selected Features:", len(selected))

reduction = (
    (n_features - len(selected)) / n_features
) * 100

print("Feature Reduction: {:.2f}%".format(reduction))

# 6. Final Random Forest Classification

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train[:, selected], y_train)

pred = model.predict(X_test[:, selected])

# 7. Performance Metrics

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred, zero_division=0)
f1 = f1_score(y_test, pred, zero_division=0)

tn, fp, fn, tp = confusion_matrix(
    y_test, pred, labels=[0, 1]
).ravel()

fpr = fp / (fp + tn) if (fp + tn) else 0

runtime = time.time() - start

# 8. Display Results

print("\n========== FINAL RESULTS ==========")

print("Accuracy :", round(accuracy, 4))
print("Precision :", round(precision, 4))
print("Recall :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("False Positive Rate :", round(fpr, 4))

print("\nConfusion Matrix:")
print([[tn, fp], [fn, tp]])

print("\nTrue Positives :", tp)
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)

print("\nTotal Runtime:", round(runtime, 2), "seconds")

print("\nIntrusion Detection Completed!")


Samples: 3000
Features: 15

Starting Binary Cuckoo Search...

Iteration 1/15 | Best Fitness: 0.14711 | Selected Features: 12
Iteration 2/15 | Best Fitness: 0.12936 | Selected Features: 10
Iteration 3/15 | Best Fitness: 0.12936 | Selected Features: 10
Iteration 4/15 | Best Fitness: 0.12936 | Selected Features: 10
Iteration 5/15 | Best Fitness: 0.12936 | Selected Features: 10
Iteration 6/15 | Best Fitness: 0.12936 | Selected Features: 10
Iteration 7/15 | Best Fitness: 0.12936 | Selected Features: 10
Iteration 8/15 | Best Fitness: 0.12936 | Selected Features: 10
Iteration 9/15 | Best Fitness: 0.12936 | Selected Features: 10
Iteration 10/15 | Best Fitness: 0.12936 | Selected Features: 10
Iteration 11/15 | Best Fitness: 0.12936 | Selected Features: 10
Iteration 12/15 | Best Fitness: 0.12936 | Selected Features: 10
Iteration 13/15 | Best Fitness: 0.12288 | Selected Features: 9
Iteration 14/15 | Best Fitness: 0.12288 | Selected Features: 9
Iteration 15/15 | Best Fitness: 0.12288 | Selected Fe